# Clase 028 — groupby (split-apply-combine)

**Parte 0** · VanderPlas cap. 3 § 3.9.

> 🎯 El patrón fundamental del análisis tabular. 4 métodos: agg, transform, filter, apply.

> ⏱️ ~90 min

## ⚙️ Setup

In [ ]:
import numpy as np
import pandas as pd
rng = np.random.default_rng(42)

# Mini-dataset penguin-like
df = pd.DataFrame({
    'species': ['Adelie']*5 + ['Chinstrap']*4 + ['Gentoo']*5,
    'sex'    : ['M','F','M','F','M', 'M','F','M','F',  'M','F','M','F','M'],
    'masa'   : [3750, 3800, 3650, 3900, 3700,  3500, 3400, 3600, 3550,  5050, 4800, 5200, 4900, 5100],
    'pico'   : [39.1, 39.5, 40.3, 38.8, 39.3,  46.5, 46.0, 46.8, 45.9,  48.6, 47.5, 49.0, 48.2, 48.8],
})
print(df.head())

## 1️⃣ Split-apply-combine

```
split:  divide el DataFrame por valores de una columna
apply:  aplica función a cada grupo
combine: junta los resultados
```

El objeto `GroupBy` no calcula nada hasta que llamas una operación (lazy):

In [ ]:
g = df.groupby('species')
print(f'tipo: {type(g).__name__}')
print(f'grupos: {list(g.groups.keys())}')
print(f'tamaño por grupo:')
print(g.size())

## 2️⃣ `agg` — reduce a una fila por grupo

In [ ]:
# Una sola función
print('media por species:')
print(g[['masa','pico']].mean().round(2))

# Dict de funciones distintas
print('\nagg con dict:')
print(g.agg({'masa': 'mean', 'pico': ['min','max']}).round(2))

# Funciones nombradas (named aggregation)
print('\nnamed aggregation:')
print(g.agg(
    masa_media=('masa', 'mean'),
    pico_max=('pico', 'max'),
    n=('masa', 'count'),
).round(2))

## 3️⃣ `transform` — misma shape, broadcast por grupo

Útil para crear features dentro de un grupo (z-score, ratio sobre el grupo, imputación).

In [ ]:
# z-score de masa POR ESPECIE
df['masa_z'] = g['masa'].transform(lambda s: (s - s.mean()) / s.std())
print(df.round(3))

# Verifica: cada grupo tiene media ≈ 0 y std ≈ 1
print('\nMedia z por species:')
print(df.groupby('species')['masa_z'].mean().round(3))
print('\nStd z por species:')
print(df.groupby('species')['masa_z'].std().round(3))

## 4️⃣ `filter` — conserva grupos completos

In [ ]:
# Solo species con más de 4 individuos
result = g.filter(lambda x: len(x) > 4)
print(result['species'].value_counts())

## 5️⃣ `apply` — flexible y lento

Úsalo cuando los 3 anteriores no alcanzan (típicamente cuando necesitas devolver un DataFrame por grupo).

In [ ]:
# El más pesado de cada species
top = g.apply(lambda x: x.nlargest(1, 'masa'), include_groups=False)
print(top)

## 6️⃣ Múltiples columnas de agrupación

In [ ]:
by_sex = df.groupby(['species', 'sex'])['masa'].mean().round(0)
print('media por species × sex (MultiIndex):')
print(by_sex)
print('\nunstack(sex) → wide:')
print(by_sex.unstack('sex'))

## 🧠 Cuándo cada método

| Método | Shape salida | Caso típico |
|---|---|---|
| `agg` | filas = #grupos | resumen estadístico |
| `transform` | filas = original | z-score, normalizar por grupo |
| `filter` | subset de original | excluir grupos pequeños/raros |
| `apply` | flexible | cuando los otros 3 no alcanzan |

## ✅ Checklist

- [ ] Entiendo split-apply-combine
- [ ] Uso named aggregation con `agg(...)`
- [ ] Sé cuándo `transform` (preserva shape) vs `agg` (reduce)
- [ ] Uso `filter` para excluir grupos enteros
- [ ] Reservo `apply` para casos que los 3 anteriores no resuelven

## 📝 Homework

Ver `README.md`. agg múltiple, transform z-score, filter por n, apply top-3.

## 📖 Definiciones y características

**Split-apply-combine**

Patrón: (1) **split** divide datos por valor de columna(s) → grupos; (2) **apply** ejecuta función en cada grupo; (3) **combine** junta resultados. El más usado en análisis tabular.

**`agg` (= aggregate)**

Reduce cada grupo a una fila (sum, mean, count, std). Acepta función nombrada, lista de funciones, o dict por columna: `agg({'a': 'sum', 'b': ['min','max']})`.

**`transform`**

Aplica función por grupo PERO mantiene la shape original (broadcastea resultado a cada fila). Ideal para z-score por grupo, imputación por grupo, ratios.

**`filter` (groupby)**

Filtra **grupos completos** (no filas) según una condición. `g.filter(lambda x: len(x) > 100)` mantiene solo grupos con >100 filas.

**`apply` (groupby)**

El más flexible y el más lento. Cualquier función custom por grupo (puede devolver Series, DataFrame, escalar). Úsalo solo cuando agg/transform/filter no alcanzan.

**Named aggregation**

Sintaxis pandas 0.25+: `g.agg(total=('monto', 'sum'), n=('id', 'count'))`. Más legible que el dict tradicional, permite renombrar en el mismo paso.

## ⚠️ Errores comunes

| Síntoma / mensaje | Causa y cómo arreglar |
|---|---|
| `g.apply(...)` lanza FutureWarning sobre `include_groups` | Pandas 2.2+ cambia comportamiento. **Fix**: `g.apply(func, include_groups=False)` para que la función no reciba la columna de groupby. |
| `g.mean()` solo muestra cols numéricas | Comportamiento intencional (pandas 2+). **Fix**: `g.mean(numeric_only=True)` para silenciar warning, o selecciona cols explícito: `g[['a','b']].mean()`. |
| `g['col'].transform(...)` da error "function did not transform" | Tu función devolvió shape distinta a la entrada. **Fix**: `transform` requiere shape igual. Usa `apply` si necesitas más libertad. |
| Resultado de `g.agg(...)` tiene MultiIndex en columnas y es engorroso | Lista de funciones por columna → MultiIndex automático. **Fix**: usa named aggregation: `g.agg(total=('x', 'sum'))`. |
| `groupby(col).size()` vs `count()` dan resultados distintos | **`size`**: número de filas por grupo (incluye NaN). **`count`**: número de NON-NaN por columna. Para 'cuántas filas hay', siempre `size()`. |

## ❓ Preguntas frecuentes

**❓ ¿`agg`, `transform`, `filter` o `apply`?**

**`agg`**: reduces a 1 fila por grupo (resumen). **`transform`**: mantienes shape original (z-score). **`filter`**: excluyes grupos enteros. **`apply`**: lo demás, asumiendo overhead.

**❓ ¿Cómo agrego columnas usando agg + named?**

`g.agg(total=('monto', 'sum'), avg=('monto', 'mean'), n=('id', 'count')).reset_index()`. Tres columnas nombradas en una sola operación.

**❓ ¿`groupby([a, b]).agg(...).reset_index()` o `as_index=False`?**

Equivalentes. `as_index=False` evita el `reset_index()` posterior. Para encadenar con merge/concat, `as_index=False` es más limpio.

**❓ ¿Cómo agrupo por una expresión derivada?**

Pasa Series directa: `df.groupby(df['fecha'].dt.year)`. O crea columna temporal: `df.groupby(df['fecha'].dt.year.rename('año'))`.

**❓ ¿groupby es lento con miles de grupos?**

Con N=1M filas y K=1000 grupos, debería ser <1s. Si es más lento, posibles causas: agg con función custom Python (no built-in), keys con dtype `object` (string), o falta de sort. Usa `sort=False` si no necesitas orden.

## 🔗 Referencias

- VanderPlas cap. 3 § 3.9
- [pandas groupby](https://pandas.pydata.org/docs/user_guide/groupby.html)
- Wickham, *Split-apply-combine* (2011)

➡️ **Siguiente:** [029 — pivot tables y crosstab](../029-pandas-pivot-tables-y-crosstab/README.md)

## ✅ Soluciones de los ejercicios

A continuación, cada ejercicio de la sección `🧪 Ejercicios` del README resuelto y comentado. Todo el código es **ejecutable sin conexión** (datos sintéticos) e incluye `assert`/`print` para que compruebes el resultado. Intenta resolverlos por tu cuenta antes de mirar la solución.

**Ej. 1 — Agg básico:** media por especie.

In [ ]:
import numpy as np, pandas as pd

def make_penguins(seed=42, with_na=False):
    """DataFrame sintetico estilo Palmer Penguins (344 filas), sin internet."""
    rng = np.random.default_rng(seed)
    cfg = {  # especie: (n, islas, bill_len, bill_depth, flipper, body_mass)
        'Adelie':    (152, ['Torgersen', 'Biscoe', 'Dream'], 38.8, 18.3, 190, 3700),
        'Chinstrap': (68,  ['Dream'],                        48.8, 18.4, 196, 3733),
        'Gentoo':    (124, ['Biscoe'],                       47.5, 15.0, 217, 5076),
    }
    filas = []
    for sp, (n, islas, bl, bd, fl, bm) in cfg.items():
        for _ in range(n):
            sex = rng.choice(['male', 'female'])
            k = 1.0 if sex == 'male' else 0.93
            filas.append({
                'species': sp,
                'island': rng.choice(islas),
                'bill_length_mm': round(float(rng.normal(bl, 2.5)), 1),
                'bill_depth_mm': round(float(rng.normal(bd, 1.2)), 1),
                'flipper_length_mm': float(round(rng.normal(fl, 6))),
                'body_mass_g': float(round(rng.normal(bm * k, 300))),
                'sex': sex,
            })
    df = pd.DataFrame(filas)
    if with_na:
        idx = rng.choice(df.index, size=12, replace=False)
        df.loc[idx[:6], 'bill_length_mm'] = np.nan
        df.loc[idx[6:], 'sex'] = np.nan
    return df

df = make_penguins()
medias = df.groupby('species').mean(numeric_only=True)
print(medias.round(1))
assert medias.shape[0] == 3

**Ej. 2 — Agg con nombres** (bill mean, mass max, count).

In [ ]:
resumen = df.groupby('species').agg(
    bill_medio=('bill_length_mm', 'mean'),
    masa_max=('body_mass_g', 'max'),
    n=('species', 'count'))
print(resumen.round(1))
assert resumen['n'].sum() == 344

**Ej. 3 — Transform:** z-score de la masa dentro de cada especie.

In [ ]:
import numpy as np
g = df.groupby('species')['body_mass_g']
df['mass_z'] = (df['body_mass_g'] - g.transform('mean')) / g.transform('std')
assert abs(df.groupby('species')['mass_z'].mean()).max() < 1e-9   # media ~0 por grupo
print(df.groupby('species')['mass_z'].agg(['mean', 'std']).round(3))

**Ej. 4 — Filter:** solo especies con >100 individuos.

In [ ]:
grandes = df.groupby('species').filter(lambda x: len(x) > 100)
print('especies conservadas:', sorted(grandes['species'].unique()))
assert set(grandes['species'].unique()) == {'Adelie', 'Gentoo'}   # Chinstrap (68) excluida

**Ej. 5 — El individuo más pesado de cada especie.**

In [ ]:
top = df.loc[df.groupby('species')['body_mass_g'].idxmax()]
print(top[['species', 'body_mass_g']])
assert top.shape[0] == 3 and top['species'].nunique() == 3